# 03 — Scaled GPT training

In [1]:
import os, time, math
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

In [2]:
train_data = np.memmap("data/train.bin", dtype=np.uint16, mode="r")
val_data   = np.memmap("data/val.bin",   dtype=np.uint16, mode="r")

In [3]:
# ---------------- model ----------------
block_size = 256
vocab_size = 4096
n_embd     = 384
n_head     = 6          # head_size = 64
n_layer    = 6
dropout    = 0.0        # train ~= val, no overfitting to fight
assert n_embd % n_head == 0, "n_embd must be divisible by n_head"

# ---------------- training ----------------
batch_size     = 32     # raise to 64 if GPU memory allows
max_iters      = 24400    # SMOKE TEST first; then ~12000 (see tokens/sec)
eval_interval  = 500
eval_iters     = 50     # each eval step is now much heavier than before
learning_rate  = 6e-4   # peak LR
min_lr         = 6e-5   # LR at the end of cosine decay (10% of peak)
warmup_iters   = 200
lr_decay_iters = max_iters
grad_clip      = 1.0
ckpt_dir       = "model"

device = "cuda" if torch.cuda.is_available() else "cpu"

# Mixed precision: bf16 on Ampere (RTX 30xx) and newer, otherwise fp16 + GradScaler
if device == "cuda" and torch.cuda.is_bf16_supported():
    amp_dtype = torch.bfloat16
elif device == "cuda":
    amp_dtype = torch.float16
else:
    amp_dtype = torch.float32
use_scaler = (amp_dtype == torch.float16)
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")
print("amp dtype:", amp_dtype, "| GradScaler:", use_scaler)

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy(data[i : i + block_size].astype(np.int64))         for i in ix.tolist()])
    y = torch.stack([torch.from_numpy(data[i + 1 : i + 1 + block_size].astype(np.int64)) for i in ix.tolist()])
    return x.to(device), y.to(device)

device: cuda NVIDIA GeForce RTX 3050 4GB Laptop GPU
amp dtype: torch.bfloat16 | GradScaler: False


In [4]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)                                           # (B, T, hs)
        q = self.query(x)                                         # (B, T, hs)
        v = self.value(x)                                         # (B, T, hs)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5       # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)                              # rows sum to 1
        wei = self.dropout(wei)
        return wei @ v                                            # (B, T, hs)


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList(Head(head_size) for _ in range(num_heads))
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)       # (B, T, C)
        out = self.dropout(self.proj(out))                        # FIX: dropout was defined but unused
        return out


class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """Communication (attention) followed by computation (MLP), pre-norm + residuals."""
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.sa   = MultiHeadAttention(n_head, n_embd // n_head)
        self.ffwd = FeedForward(n_embd)
        self.ln1  = nn.LayerNorm(n_embd)
        self.ln2  = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [5]:
class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table      = nn.Embedding(vocab_size, n_embd)
        self.positional_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks  = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f    = nn.LayerNorm(n_embd)                       # NEW: final LayerNorm
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

        # NEW: weight tying -- input embedding and output projection share one matrix
        self.token_embedding_table.weight = self.lm_head.weight

        # NEW: GPT-2 style init (small normal weights, zero biases)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape                                                     # (B, T)
        tok_emb = self.token_embedding_table(idx)                            # (B, T, C)
        pos_emb = self.positional_embedding_table(torch.arange(T, device=idx.device))  # (T, C)
        x = tok_emb + pos_emb                                                # (B, T, C)
        x = self.blocks(x)                                                   # (B, T, C)
        x = self.ln_f(x)                                                     # (B, T, C)
        logits = self.lm_head(x)                                             # (B, T, V)

        loss = None
        if targets is not None:
            B, T, V = logits.shape
            loss = F.cross_entropy(logits.view(B * T, V), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits.float(), dim=-1)
            next_id = torch.multinomial(probs, 1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

In [6]:
model = GPTLanguageModel().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, betas=(0.9, 0.95), weight_decay=0.1)
scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)

# model.parameters() de-duplicates the tied matrix, so it is counted once
n_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {n_params/1e6:.2f}M   (hand estimate: V*d + 12*L*d^2 + T*d)")
print(f"hand estimate: {(vocab_size*n_embd + 12*n_layer*n_embd**2 + block_size*n_embd)/1e6:.2f}M")

parameters: 12.31M   (hand estimate: V*d + 12*L*d^2 + T*d)
hand estimate: 12.29M


In [7]:
def get_lr(it):
    # 1) linear warmup
    if it < warmup_iters:
        return learning_rate * (it + 1) / warmup_iters
    # 2) after decay window: stay at min_lr
    if it > lr_decay_iters:
        return min_lr
    # 3) cosine decay from learning_rate down to min_lr
    ratio = (it - warmup_iters) / max(1, lr_decay_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * ratio))      # goes 1 -> 0
    return min_lr + coeff * (learning_rate - min_lr)


@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch(split)
            with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=(device == "cuda")):
                _, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


def save_checkpoint(path, step, val_loss):
    torch.save({
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "step": step,
        "val_loss": val_loss,
        "config": dict(block_size=block_size, vocab_size=vocab_size, n_embd=n_embd,
                       n_head=n_head, n_layer=n_layer, dropout=dropout),
    }, path)

In [8]:
os.makedirs(ckpt_dir, exist_ok=True)
best_val = float("inf")
tokens_per_step = batch_size * block_size
t0 = time.time()

for step in range(max_iters):

    # set the learning rate for this step
    lr = get_lr(step)
    for g in optimizer.param_groups:
        g["lr"] = lr

    # evaluate + checkpoint
    if step % eval_interval == 0 or step == max_iters - 1:
        losses = estimate_loss()
        print(f"step {step}: train {losses['train']:.4f}, val {losses['val']:.4f}, lr {lr:.2e}")
        if losses["val"] < best_val:
            best_val = losses["val"]
            save_checkpoint(os.path.join(ckpt_dir, "gpt_best.pt"), step, best_val)
        t0 = time.time()                       # don't count eval time in tokens/sec

    x, y = get_batch("train")

    # forward in mixed precision
    with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=(device == "cuda")):
        logits, loss = model(x, y)

    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()              # no-op scaling when the scaler is disabled (bf16)
    scaler.unscale_(optimizer)                 # unscale BEFORE clipping, so the clip sees real gradient sizes
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    scaler.step(optimizer)
    scaler.update()

    # throughput every 50 steps
    if step % 50 == 0 and step > 0:
        if device == "cuda":
            torch.cuda.synchronize()           # GPU runs async: wait for it before reading the clock
        dt = time.time() - t0
        steps_done = (step % eval_interval) or eval_interval
        print(f"  step {step}: loss {loss.item():.4f}, {steps_done * tokens_per_step / dt:,.0f} tokens/sec")

save_checkpoint(os.path.join(ckpt_dir, "gpt_last.pt"), max_iters, best_val)
print("best val:", best_val)

step 0: train 8.3717, val 8.3705, lr 3.00e-06
  step 50: loss 6.1387, 23,836 tokens/sec
  step 100: loss 4.9631, 24,175 tokens/sec
  step 150: loss 4.2582, 23,608 tokens/sec
  step 200: loss 3.9341, 23,643 tokens/sec
  step 250: loss 3.7673, 23,641 tokens/sec
  step 300: loss 3.6246, 23,611 tokens/sec
  step 350: loss 3.3568, 23,599 tokens/sec
  step 400: loss 3.3298, 23,585 tokens/sec
  step 450: loss 3.1876, 23,571 tokens/sec
step 500: train 3.0972, val 3.1291, lr 6.00e-04
  step 500: loss 2.9680, 11,651,429 tokens/sec
  step 550: loss 3.0482, 22,332 tokens/sec
  step 600: loss 2.9032, 22,543 tokens/sec
  step 650: loss 2.9515, 22,570 tokens/sec
  step 700: loss 2.8025, 22,576 tokens/sec
  step 750: loss 2.9194, 22,390 tokens/sec
  step 800: loss 2.6588, 22,410 tokens/sec
  step 850: loss 2.8419, 22,430 tokens/sec
  step 900: loss 2.5644, 22,447 tokens/sec
  step 950: loss 2.6024, 22,487 tokens/sec
step 1000: train 2.6035, val 2.6171, lr 5.99e-04
  step 1000: loss 2.5971, 11,434,296 

In [9]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("tokenizer")
EOT_ID = tok.convert_tokens_to_ids("<|endoftext|>")   # 0

model.eval()
start = torch.tensor([[EOT_ID]], dtype=torch.long, device=device)   # (1, 1)
out = model.generate(start, max_new_tokens=200)                     # (1, 201)
print(tok.decode(out[0].tolist()))

<|endoftext|>Ben and Lily are twins. They like to play outside. They have a big wagon. They load the wagon with toys. Theyump the toys in the cart.

"Look, Ben, I have a red car!" Lily says to her friends.

"Wow, soldiers, toys are fun!" Ben says. He is very smart.

They put the toys on the log. They make it go vroom vroom. They are happy.

But soon they start to argue. They both want the toys. They pull the wagon. The wagon is broken. They cry.

" elderly friend, don't fight, they are friends again!" Lily says.

"Sorry, Lily, I'm sorry for the toys?" Ben says.

They hug each other. They decide to share the toys. They put the cars on the low bottom. They make the boat fight again.

They use the other toys to play together


In [10]:
ckpt = torch.load("model/gpt_best.pt", map_location=device)
model.load_state_dict(ckpt["model"])
optimizer.load_state_dict(ckpt["optimizer"])
scaler.load_state_dict(ckpt["scaler"])
start_step = ckpt["step"]